# Translate Job Titles from Vietnamese to English

**Purpose:** Translate only the `title` field from Vietnamese to English and update the database

**Note:** Run cells in order

## 1. Install Required Libraries

In [26]:
!pip install pymysql sqlalchemy deep-translator langdetect pandas tqdm cryptography

## 2. Import Libraries

In [27]:
import pymysql
import pandas as pd
import time
import ssl
import tempfile
import os
from sqlalchemy import create_engine, text
from deep_translator import GoogleTranslator
from langdetect import detect, LangDetectException
from tqdm.notebook import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
print('✅ Import OK')

✅ Import OK


## 3. Database Configuration and SSL Certificate

**IMPORTANT:** 
- Update database credentials
- Copy your .pem certificate content to CA_CERT_CONTENT

In [28]:
# ========== DATABASE CONFIGURATION ==========
DB_CONFIG = {
    'host': 'gateway01.ap-southeast-1.prod.aws.tidbcloud.com',
    'port': 4000,
    'user': '4GJhpnEevqoZfyD.root',
    'password': 'oiK2dgnVVJLVHL4v',
    'database': 'data-mining',
    'charset': 'utf8mb4'
}

# ========== PASTE CERTIFICATE CONTENT ==========
CA_CERT_CONTENT = """
-----BEGIN CERTIFICATE-----
MIIFazCCA1OgAwIBAgIRAIIQz7DSQONZRGPgu2OCiwAwDQYJKoZIhvcNAQELBQAw
TzELMAkGA1UEBhMCVVMxKTAnBgNVBAoTIEludGVybmV0IFNlY3VyaXR5IFJlc2Vh
cmNoIEdyb3VwMRUwEwYDVQQDEwxJU1JHIFJvb3QgWDEwHhcNMTUwNjA0MTEwNDM4
WhcNMzUwNjA0MTEwNDM4WjBPMQswCQYDVQQGEwJVUzEpMCcGA1UEChMgSW50ZXJu
ZXQgU2VjdXJpdHkgUmVzZWFyY2ggR3JvdXAxFTATBgNVBAMTDElTUkcgUm9vdCBY
MTCCAiIwDQYJKoZIhvcNAQEBBQADggIPADCCAgoCggIBAK3oJHP0FDfzm54rVygc
h77ct984kIxuPOZXoHj3dcKi/vVqbvYATyjb3miGbESTtrFj/RQSa78f0uoxmyF+
0TM8ukj13Xnfs7j/EvEhmkvBioZxaUpmZmyPfjxwv60pIgbz5MDmgK7iS4+3mX6U
A5/TR5d8mUgjU+g4rk8Kb4Mu0UlXjIB0ttov0DiNewNwIRt18jA8+o+u3dpjq+sW
T8KOEUt+zwvo/7V3LvSye0rgTBIlDHCNAymg4VMk7BPZ7hm/ELNKjD+Jo2FR3qyH
B5T0Y3HsLuJvW5iB4YlcNHlsdu87kGJ55tukmi8mxdAQ4Q7e2RCOFvu396j3x+UC
B5iPNgiV5+I3lg02dZ77DnKxHZu8A/lJBdiB3QW0KtZB6awBdpUKD9jf1b0SHzUv
KBds0pjBqAlkd25HN7rOrFleaJ1/ctaJxQZBKT5ZPt0m9STJEadao0xAH0ahmbWn
OlFuhjuefXKnEgV4We0+UXgVCwOPjdAvBbI+e0ocS3MFEvzG6uBQE3xDk3SzynTn
jh8BCNAw1FtxNrQHusEwMFxIt4I7mKZ9YIqioymCzLq9gwQbooMDQaHWBfEbwrbw
qHyGO0aoSCqI3Haadr8faqU9GY/rOPNk3sgrDQoo//fb4hVC1CLQJ13hef4Y53CI
rU7m2Ys6xt0nUW7/vGT1M0NPAgMBAAGjQjBAMA4GA1UdDwEB/wQEAwIBBjAPBgNV
HRMBAf8EBTADAQH/MB0GA1UdDgQWBBR5tFnme7bl5AFzgAiIyBpY9umbbjANBgkq
hkiG9w0BAQsFAAOCAgEAVR9YqbyyqFDQDLHYGmkgJykIrGF1XIpu+ILlaS/V9lZL
ubhzEFnTIZd+50xx+7LSYK05qAvqFyFWhfFQDlnrzuBZ6brJFe+GnY+EgPbk6ZGQ
3BebYhtF8GaV0nxvwuo77x/Py9auJ/GpsMiu/X1+mvoiBOv/2X/qkSsisRcOj/KK
NFtY2PwByVS5uCbMiogziUwthDyC3+6WVwW6LLv3xLfHTjuCvjHIInNzktHCgKQ5
ORAzI4JMPJ+GslWYHb4phowim57iaztXOoJwTdwJx4nLCgdNbOhdjsnvzqvHu7Ur
TkXWStAmzOVyyghqpZXjFaH3pO3JLF+l+/+sKAIuvtd7u+Nxe5AW0wdeRlN8NwdC
jNPElpzVmbUq4JUagEiuTDkHzsxHpFKVK7q4+63SM1N95R1NbdWhscdCb+ZAJzVc
oyi3B43njTOQ5yOf+1CceWxG1bQVs5ZufpsMljq4Ui0/1lvh+wjChP4kqKOJ2qxq
4RgqsahDYVvTH9w7jXbyLeiNdd8XM2w9U/t7y0Ff/9yi0GE44Za4rF2LN9d11TPA
mRGunUHBcnWEvgJBQl9nJEiU0Zsnvgc/ubhPgXRR4Xq37Z0j4r7g1SgEEzwxA57d
emyPxgcYxn/eR44/KJ4EBs+lVDR3veyJm+kXQ99b21/+jh5Xos1AnX5iItreGCc=
-----END CERTIFICATE-----
""".strip()

print(f'📊 Database: {DB_CONFIG["database"]}')
print(f'🔗 Host: {DB_CONFIG["host"]}:{DB_CONFIG["port"]}')
print(f'📜 CA Cert: {"✅" if len(CA_CERT_CONTENT) > 100 else "❌ Not configured"}')

📊 Database: data-mining
🔗 Host: gateway01.ap-southeast-1.prod.aws.tidbcloud.com:4000
📜 CA Cert: ✅


## 4. Create SSL Connection

In [29]:
temp_files = []
try:
    ssl_context = ssl.create_default_context()
    
    if CA_CERT_CONTENT:
        ca_temp = tempfile.NamedTemporaryFile(mode='w', suffix='.pem', delete=False)
        ca_temp.write(CA_CERT_CONTENT)
        ca_temp.close()
        temp_files.append(ca_temp.name)
        ssl_context.load_verify_locations(ca_temp.name)
        print('✅ Loaded CA certificate')
    
    DATABASE_URL = f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}?charset={DB_CONFIG['charset']}"
    engine = create_engine(DATABASE_URL, connect_args={'ssl': ssl_context}, pool_pre_ping=True)
    
    print('\n🔄 Testing connection...')
    with engine.connect() as conn:
        result = conn.execute(text('SELECT COUNT(*) FROM jobs'))
        total = result.fetchone()[0]
        print(f'\n✅ Connection successful!')
        print(f'📊 Total jobs: {total:,}')
        
except Exception as e:
    print(f'❌ Error: {e}')
    for f in temp_files:
        try: os.unlink(f)
        except: pass
    raise

✅ Loaded CA certificate

🔄 Testing connection...

✅ Connection successful!
📊 Total jobs: 12,117


## 5. Define Translation Helper

In [30]:
class TitleTranslator:
    def __init__(self):
        self.translator = GoogleTranslator(source='vi', target='en')
        self.cache = {}
        self.stats = {
            'translated': 0,
            'already_english': 0,
            'errors': 0,
            'cached': 0
        }
    
    def is_vietnamese(self, text):
        """Check if text is in Vietnamese"""
        if not text or len(str(text).strip()) < 3:
            return False
        
        try:
            lang = detect(str(text))
            return lang == 'vi'
        except:
            return False
    
    def translate_title(self, title, max_retries=3):
        """Translate title from Vietnamese to English"""
        if not title or not str(title).strip():
            return title
        
        title_str = str(title).strip()
        
        # Check cache first
        if title_str in self.cache:
            self.stats['cached'] += 1
            return self.cache[title_str]
        
        # Check if Vietnamese
        if not self.is_vietnamese(title_str):
            self.stats['already_english'] += 1
            self.cache[title_str] = title_str
            return title_str
        
        # Translate with retry
        for attempt in range(max_retries):
            try:
                result = self.translator.translate(title_str)
                
                if result and result.strip():
                    self.stats['translated'] += 1
                    self.cache[title_str] = result
                    time.sleep(0.1)  # Rate limiting
                    return result
                else:
                    return title_str
                    
            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(1 * (attempt + 1))
                else:
                    print(f'⚠️ Translation error: {str(e)[:50]}')
                    self.stats['errors'] += 1
                    return title_str
        
        return title_str
    
    def print_stats(self):
        """Print translation statistics"""
        print(f"📊 Translated: {self.stats['translated']:,} | "
              f"Already English: {self.stats['already_english']:,} | "
              f"Cached: {self.stats['cached']:,} | "
              f"Errors: {self.stats['errors']:,}")

print('✅ TitleTranslator class defined')
print('🌐 Will detect Vietnamese and translate to English')
print('⚡ Includes caching for better performance')

✅ TitleTranslator class defined
🌐 Will detect Vietnamese and translate to English
⚡ Includes caching for better performance


## 6. Load Jobs from Database

In [31]:
print('📥 Loading jobs from database...')
query = 'SELECT id, title, source FROM jobs ORDER BY id'
df = pd.read_sql(query, engine)
print(f'✅ Loaded {len(df):,} jobs')

# Show sample titles
print(f'\n📝 Sample titles (first 5):')
for idx, row in df.head().iterrows():
    print(f"  {row['id']}: {row['title']}")

# Count by source
if 'source' in df.columns:
    print(f'\n📊 Jobs by source:')
    for source, count in df['source'].value_counts().items():
        print(f'  • {source}: {count:,}')

📥 Loading jobs from database...
✅ Loaded 12,117 jobs

📝 Sample titles (first 5):
  3462: Frontend developer (PA project)
  3463: QA Engineer (Automotive)
  3464: Java Developer
  3465: Publishing staff - Webtoon/Manga - Full-time
  3466: Webtoon Colorist - Colorist - Full-time

📊 Jobs by source:
  • linkedin: 9,132
  • topcv: 1,941
  • itviec: 743
  • topdev: 301


## 7. Translate Titles

In [32]:
translator = TitleTranslator()
df_translated = df.copy()

print(f'🚀 Starting translation of {len(df):,} job titles')
print(f'⏱️  Estimated time: {len(df)*0.15/60:.1f} minutes\n')

start_time = time.time()

for idx in tqdm(range(len(df_translated)), desc='Translating titles'):
    row = df_translated.iloc[idx]
    
    if pd.notna(row['title']):
        original_title = str(row['title'])
        translated_title = translator.translate_title(original_title)
        df_translated.at[idx, 'title'] = translated_title
    
    # Progress update every 100 jobs
    if (idx + 1) % 100 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(df_translated) - idx - 1) / rate
        print(f'\n[{idx+1}/{len(df_translated)}] Time: {elapsed/60:.1f}m | '
              f'Remaining: {remaining/60:.1f}m | Rate: {rate:.1f} jobs/s')
        translator.print_stats()

elapsed_time = time.time() - start_time
print(f'\n✅ Translation complete!')
print(f'⏱️  Total time: {elapsed_time/60:.1f} minutes ({elapsed_time:.0f} seconds)')
print(f'⚡ Average rate: {len(df)/elapsed_time:.2f} jobs/second\n')
translator.print_stats()

🚀 Starting translation of 12,117 job titles
⏱️  Estimated time: 30.3 minutes



Translating titles:   0%|          | 0/12117 [00:00<?, ?it/s]


[100/12117] Time: 0.0m | Remaining: 2.2m | Rate: 92.7 jobs/s
📊 Translated: 0 | Already English: 88 | Cached: 12 | Errors: 0

[200/12117] Time: 0.0m | Remaining: 2.8m | Rate: 70.6 jobs/s
📊 Translated: 1 | Already English: 169 | Cached: 30 | Errors: 0

[300/12117] Time: 0.1m | Remaining: 3.6m | Rate: 55.4 jobs/s
📊 Translated: 4 | Already English: 262 | Cached: 34 | Errors: 0

[400/12117] Time: 0.1m | Remaining: 3.1m | Rate: 64.0 jobs/s
📊 Translated: 4 | Already English: 360 | Cached: 36 | Errors: 0

[500/12117] Time: 0.1m | Remaining: 2.8m | Rate: 69.8 jobs/s
📊 Translated: 4 | Already English: 460 | Cached: 36 | Errors: 0

[600/12117] Time: 0.1m | Remaining: 2.5m | Rate: 75.3 jobs/s
📊 Translated: 4 | Already English: 557 | Cached: 39 | Errors: 0

[700/12117] Time: 0.1m | Remaining: 2.4m | Rate: 79.7 jobs/s
📊 Translated: 4 | Already English: 653 | Cached: 43 | Errors: 0

[800/12117] Time: 0.2m | Remaining: 2.2m | Rate: 84.1 jobs/s
📊 Translated: 4 | Already English: 748 | Cached: 48 | Err

## 8. Preview Results

In [33]:
print('🔍 TRANSLATION COMPARISON (Before → After)\n')
print('='*100)

# Find titles that were translated (changed)
changed_mask = df['title'] != df_translated['title']
changed_count = changed_mask.sum()

print(f'📊 Summary:')
print(f'  • Total jobs: {len(df):,}')
print(f'  • Titles changed: {changed_count:,}')
print(f'  • Titles unchanged: {(~changed_mask).sum():,}\n')

if changed_count > 0:
    print('📝 Sample translations (first 10 changed):\n')
    changed_df = df[changed_mask].head(10)
    
    for idx in changed_df.index:
        orig = df.loc[idx, 'title']
        trans = df_translated.loc[idx, 'title']
        job_id = df.loc[idx, 'id']
        print(f'Job {job_id}:')
        print(f'  Before: {orig}')
        print(f'  After:  {trans}')
        print()
else:
    print('ℹ️  No titles were changed (all already in English or translation failed)')

print('='*100)

🔍 TRANSLATION COMPARISON (Before → After)

📊 Summary:
  • Total jobs: 12,117
  • Titles changed: 2
  • Titles unchanged: 12,115

📝 Sample translations (first 10 changed):

Job 3674:
  Before: Frontend Developer (Middle/Senior) - Mức Lương 1000$ - 2000$
  After:  Frontend Developer (Middle/Senior) - Salary 1000$ - 2000$

Job 5073:
  Before: Thực Tập SinhJavaScript- PlayableAds(Short DemoGame)
  After:  InternJavaScript- PlayableAds(Short DemoGame)



## 9. Save Backup CSV

In [34]:
backup_file = f'job_titles_translated_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df_translated.to_csv(backup_file, index=False, encoding='utf-8-sig')
print(f'✅ Backup saved: {backup_file}')
print(f'📁 File location: {os.path.abspath(backup_file)}')

# Show file size
file_size = os.path.getsize(backup_file) / 1024
print(f'💾 File size: {file_size:.1f} KB')

✅ Backup saved: job_titles_translated_20260202_162828.csv
📁 File location: /content/job_titles_translated_20260202_162828.csv
💾 File size: 572.5 KB


## 10. Update Database

In [35]:
confirm = input('⚠️  Update database with translated titles? (yes/no): ')

if confirm.lower() == 'yes':
    print('\n🔄 Updating database...')
    print('📝 Only updating titles that were translated\n')
    
    update_count = 0
    error_count = 0
    
    # Only update rows where title changed
    changed_mask = df['title'] != df_translated['title']
    changed_df = df_translated[changed_mask]
    
    print(f'📊 Will update {len(changed_df):,} jobs with translated titles')
    
    with engine.begin() as conn:
        for idx, row in tqdm(changed_df.iterrows(), total=len(changed_df), desc='Updating database'):
            try:
                update_query = '''
                    UPDATE jobs 
                    SET title = :title, updated_at = NOW()
                    WHERE id = :id
                '''
                conn.execute(text(update_query), {
                    'id': int(row['id']),
                    'title': row['title']
                })
                update_count += 1
                
            except Exception as e:
                error_count += 1
                if error_count <= 5:
                    print(f'\n⚠️ Error updating job {row["id"]}: {str(e)[:100]}')
    
    print(f'\n✅ Updated: {update_count:,} job titles')
    print(f'❌ Errors: {error_count:,}')
    
    if error_count == 0:
        # Verify updates
        print(f'\n🔍 Verifying updates...')
        check_query = 'SELECT COUNT(*) as count FROM jobs WHERE updated_at >= DATE_SUB(NOW(), INTERVAL 5 MINUTE)'
        result = pd.read_sql(check_query, engine)
        recently_updated = result['count'].iloc[0]
        print(f'✅ {recently_updated:,} jobs updated in last 5 minutes')
        
        # Show sample of updated titles from database
        sample_query = f'''
            SELECT id, title 
            FROM jobs 
            WHERE id IN ({','.join(map(str, changed_df['id'].head(3).tolist()))})
        '''
        sample_db = pd.read_sql(sample_query, engine)
        print(f'\n📝 Sample from database (first 3 updated):')
        for _, row in sample_db.iterrows():
            print(f"  Job {row['id']}: {row['title']}")
        
        print(f'\n✅ SUCCESS: Database updated successfully!')
    else:
        print(f'\n⚠️ WARNING: {error_count} errors occurred during update')
        
else:
    print('❌ Update cancelled by user')


🔄 Updating database...
📝 Only updating titles that were translated

📊 Will update 2 jobs with translated titles


Updating database:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Updated: 2 job titles
❌ Errors: 0

🔍 Verifying updates...
✅ 2 jobs updated in last 5 minutes

📝 Sample from database (first 3 updated):
  Job 3674: Frontend Developer (Middle/Senior) - Salary 1000$ - 2000$
  Job 5073: InternJavaScript- PlayableAds(Short DemoGame)

✅ SUCCESS: Database updated successfully!


## 11. Cleanup and Summary

In [36]:
# Cleanup temporary files
for f in temp_files:
    try: 
        os.unlink(f)
        print(f'🗑️  Deleted temp file: {f}')
    except: 
        pass

print('\n' + '='*80)
print('🎉 TITLE TRANSLATION COMPLETE - FINAL SUMMARY')
print('='*80)
print(f'📊 Total jobs processed: {len(df):,}')
print(f'💾 Backup file: {backup_file}')
print(f'⏱️  Processing time: {elapsed_time/60:.1f} minutes')
print()

# Translation statistics
print('📈 TRANSLATION STATISTICS:')
print(f'  ✅ Successfully translated: {translator.stats["translated"]:,}')
print(f'  💨 Cached (reused): {translator.stats["cached"]:,}')
print(f'  ⏭️  Already English: {translator.stats["already_english"]:,}')
print(f'  ❌ Errors: {translator.stats["errors"]:,}')
print(f'  📦 Cache size: {len(translator.cache):,} unique titles')
print()

# Changed vs unchanged
changed_mask = df['title'] != df_translated['title']
print('📊 CHANGES:')
print(f'  🔄 Titles changed: {changed_mask.sum():,}')
print(f'  ➖ Titles unchanged: {(~changed_mask).sum():,}')
print('='*80)

🗑️  Deleted temp file: /tmp/tmpbe_mqp44.pem

🎉 TITLE TRANSLATION COMPLETE - FINAL SUMMARY
📊 Total jobs processed: 12,117
💾 Backup file: job_titles_translated_20260202_162828.csv
⏱️  Processing time: 1.3 minutes

📈 TRANSLATION STATISTICS:
  ✅ Successfully translated: 6
  💨 Cached (reused): 4,331
  ⏭️  Already English: 7,780
  ❌ Errors: 0
  📦 Cache size: 7,786 unique titles

📊 CHANGES:
  🔄 Titles changed: 2
  ➖ Titles unchanged: 12,115


## 12. Verify All Titles Are in English

In [37]:
print('🔍 VERIFYING ALL TITLES ARE IN ENGLISH')
print('='*80)
print('📥 Loading current titles from database...\n')

# Load current titles from database
verify_query = 'SELECT id, title, source FROM jobs ORDER BY id'
df_verify = pd.read_sql(verify_query, engine)

print(f'✅ Loaded {len(df_verify):,} job titles\n')

# Check language for each title
vietnamese_titles = []
english_titles = 0
unknown_titles = 0
error_count = 0

print('🌐 Detecting language for all titles...\n')

for idx in tqdm(range(len(df_verify)), desc='Checking languages'):
    row = df_verify.iloc[idx]
    
    if pd.notna(row['title']) and str(row['title']).strip():
        title = str(row['title']).strip()
        
        # Skip very short titles
        if len(title) < 3:
            unknown_titles += 1
            continue
        
        try:
            detected_lang = detect(title)
            
            if detected_lang == 'vi':
                vietnamese_titles.append({
                    'id': row['id'],
                    'title': title,
                    'source': row['source'] if 'source' in row else 'N/A'
                })
            elif detected_lang == 'en':
                english_titles += 1
            else:
                unknown_titles += 1
                
        except Exception as e:
            error_count += 1
            if error_count <= 3:
                print(f'⚠️ Error detecting language for job {row["id"]}: {str(e)[:50]}')

# Print results
print('\n' + '='*80)
print('📊 LANGUAGE DETECTION RESULTS')
print('='*80)
print(f'✅ English titles: {english_titles:,} ({english_titles/len(df_verify)*100:.1f}%)')
print(f'🇻🇳 Vietnamese titles: {len(vietnamese_titles):,} ({len(vietnamese_titles)/len(df_verify)*100:.1f}%)')
print(f'❓ Unknown/Other: {unknown_titles:,} ({unknown_titles/len(df_verify)*100:.1f}%)')
print(f'❌ Detection errors: {error_count:,}')
print('='*80)

if len(vietnamese_titles) > 0:
    print(f'\n⚠️ WARNING: Found {len(vietnamese_titles)} Vietnamese titles that need translation!')
    print(f'\n📋 Sample Vietnamese titles (first 10):\n')
    
    for i, item in enumerate(vietnamese_titles[:10], 1):
        print(f"{i}. Job {item['id']} ({item['source']}): {item['title']}")
    
    if len(vietnamese_titles) > 10:
        print(f'\n... and {len(vietnamese_titles) - 10} more Vietnamese titles')
    
    print(f'\n💡 Recommendation: Re-run cells 7-10 to translate remaining Vietnamese titles')
else:
    print(f'\n✅ SUCCESS: All titles are in English!')
    print(f'🎉 No Vietnamese titles detected in the database')

print('\n' + '='*80)

🔍 VERIFYING ALL TITLES ARE IN ENGLISH
📥 Loading current titles from database...

✅ Loaded 12,117 job titles

🌐 Detecting language for all titles...



Checking languages:   0%|          | 0/12117 [00:00<?, ?it/s]


📊 LANGUAGE DETECTION RESULTS
✅ English titles: 8,803 (72.6%)
🇻🇳 Vietnamese titles: 5 (0.0%)
❓ Unknown/Other: 3,309 (27.3%)
❌ Detection errors: 0

⚠️ WARNING: Found 5 Vietnamese titles that need translation!

📋 Sample Vietnamese titles (first 10):

1. Job 3596 (topdev): [HN] INNOVATION CENTER - PHP PROGRAMMER
2. Job 3677 (topdev): TELECOMMUNICATION INFRASTRUCTURE MAINTENANCE TECHNICIAN (Gia LAM, LONG BIEN)
3. Job 3746 (topdev): Teller - Hanoi (Hoang Mai, Thanh Xuan, Hai Ba Trung Area)
4. Job 6234 (topcv): Kỹ SưDevopsLevel Senior
5. Job 11999 (linkedin): PENETRATION TESTER

💡 Recommendation: Re-run cells 7-10 to translate remaining Vietnamese titles



## 13. Revert Incorrect Translations

In [38]:
print('🔄 REVERT INCORRECT TRANSLATIONS')
print('='*80)
print('This tool will revert titles back to their original values from backup\n')

# Step 1: Get current database state
print('📥 Step 1: Loading current titles from database...')
current_query = 'SELECT id, title FROM jobs ORDER BY id'
df_current = pd.read_sql(current_query, engine)
print(f'✅ Loaded {len(df_current):,} current titles\n')

# Step 2: Load original data from backup CSV
print('📂 Step 2: Available backup files:')
import glob
backup_files = sorted(glob.glob('job_titles_translated_*.csv'), reverse=True)

if len(backup_files) == 0:
    print('❌ No backup files found!')
    print('   Backup files should match pattern: job_titles_translated_*.csv')
else:
    for i, file in enumerate(backup_files[:5], 1):
        file_time = os.path.getmtime(file)
        file_date = datetime.fromtimestamp(file_time).strftime('%Y-%m-%d %H:%M:%S')
        file_size = os.path.getsize(file) / 1024
        print(f'  {i}. {file} ({file_date}, {file_size:.1f} KB)')
    
    if len(backup_files) > 5:
        print(f'  ... and {len(backup_files) - 5} more backup files')
    
    print('\n📝 To revert, you need the ORIGINAL data (before translation)')
    print('   Option 1: Use backup CSV from BEFORE running translation')
    print('   Option 2: Load directly from database if translations not yet applied')
    
    backup_choice = input('\nEnter backup file name (or press Enter to load fresh from database): ').strip()
    
    if backup_choice:
        # Load from specified backup file
        if os.path.exists(backup_choice):
            df_original = pd.read_csv(backup_choice)
            print(f'\n✅ Loaded {len(df_original):,} titles from {backup_choice}')
        else:
            print(f'\n❌ File not found: {backup_choice}')
            df_original = None
    else:
        # Load original from database (assumes you have original data somewhere)
        print('\n⚠️  WARNING: Loading from current database')
        print('   This will only work if you have not yet applied translations')
        print('   or if you want to revert to current database state')
        df_original = df_current.copy()
        print(f'✅ Using current database state as "original"')
    
    if df_original is not None:
        # Step 3: Find differences
        print('\n📊 Step 3: Analyzing differences...\n')
        
        # Merge to compare
        df_compare = df_current.merge(df_original[['id', 'title']], on='id', suffixes=('_current', '_original'))
        
        # Find titles that differ
        different_mask = df_compare['title_current'] != df_compare['title_original']
        df_different = df_compare[different_mask]
        
        print(f'📈 Analysis Results:')
        print(f'  • Total jobs: {len(df_compare):,}')
        print(f'  • Titles that differ: {len(df_different):,}')
        print(f'  • Titles unchanged: {(~different_mask).sum():,}')
        
        if len(df_different) > 0:
            print(f'\n📋 Sample of differences (first 10):')
            print('='*80)
            for idx, row in df_different.head(10).iterrows():
                print(f"\nJob {row['id']}:")
                print(f"  Current:  {row['title_current']}")
                print(f"  Original: {row['title_original']}")
            
            if len(df_different) > 10:
                print(f'\n... and {len(df_different) - 10} more differences')
            
            # Step 4: Confirm and revert
            print('\n' + '='*80)
            confirm = input(f'\n⚠️  Revert {len(df_different):,} titles back to original? (yes/no): ')
            
            if confirm.lower() == 'yes':
                print('\n🔄 Reverting titles to original values...')
                
                revert_count = 0
                error_count = 0
                
                with engine.begin() as conn:
                    for idx, row in tqdm(df_different.iterrows(), total=len(df_different), desc='Reverting titles'):
                        try:
                            update_query = '''
                                UPDATE jobs 
                                SET title = :title, updated_at = NOW()
                                WHERE id = :id
                            '''
                            conn.execute(text(update_query), {
                                'id': int(row['id']),
                                'title': row['title_original']
                            })
                            revert_count += 1
                            
                        except Exception as e:
                            error_count += 1
                            if error_count <= 5:
                                print(f'\n⚠️ Error reverting job {row["id"]}: {str(e)[:100]}')
                
                print(f'\n✅ Reverted: {revert_count:,} titles')
                print(f'❌ Errors: {error_count:,}')
                
                if error_count == 0:
                    # Verify reverts
                    print(f'\n🔍 Verifying reverts...')
                    verify_query = f'''
                        SELECT id, title 
                        FROM jobs 
                        WHERE id IN ({','.join(map(str, df_different['id'].head(3).tolist()))})
                    '''
                    verify_df = pd.read_sql(verify_query, engine)
                    print(f'\n📝 Sample from database (first 3 reverted):')
                    for _, row in verify_df.iterrows():
                        original_title = df_different[df_different['id'] == row['id']]['title_original'].iloc[0]
                        match = '✅' if row['title'] == original_title else '❌'
                        print(f"  {match} Job {row['id']}: {row['title']}")
                    
                    print(f'\n✅ SUCCESS: Titles reverted successfully!')
                else:
                    print(f'\n⚠️ WARNING: {error_count} errors occurred during revert')
            else:
                print('❌ Revert cancelled by user')
        else:
            print('\nℹ️  No differences found. Current database matches original backup.')
    
print('\n' + '='*80)

🔄 REVERT INCORRECT TRANSLATIONS
This tool will revert titles back to their original values from backup

📥 Step 1: Loading current titles from database...
✅ Loaded 12,117 current titles

📂 Step 2: Available backup files:
  1. job_titles_translated_20260202_162828.csv (2026-02-02 16:28:28, 572.5 KB)
  2. job_titles_translated_20260202_162152.csv (2026-02-02 16:21:52, 572.6 KB)
  3. job_titles_translated_20260202_155527.csv (2026-02-02 15:55:27, 575.7 KB)

📝 To revert, you need the ORIGINAL data (before translation)
   Option 1: Use backup CSV from BEFORE running translation
   Option 2: Load directly from database if translations not yet applied


KeyboardInterrupt: Interrupted by user